In [2]:
import pandas as pd
import polars as pl
import numpy as np
import re 
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests

In [3]:
def gene_exonic_lengths_from_gtf( gtf_path: str) -> pd.DataFrame:
        """
        Returns a DataFrame with columns: gene_id, exonic_length
        Collapses overlapping exons per gene across all transcripts --> not necessarily 
        mane_transcript gene length
        """
        def extract_gene_id(attr):
            m = re.search(r'gene_id\s+"([^"]+)"', attr)
            if m:
                return m.group(1)
            m = re.search(r'gene_id=([^;]+)', attr)
            if m:
                return m.group(1)
            return None


        def merge_and_sum(intervals: np.ndarray) -> int:
            """
            intervals: Nx2 numpy array of [start, end] (inclusive, 1-based)
            returns total union length in bp.
            """
            if intervals.size == 0:
                return 0

            # sort by start then end
            intervals = intervals[np.lexsort((intervals[:,1], intervals[:,0]))]

            total = 0
            cur_start, cur_end = intervals[0]

            for s, e in intervals[1:]:
                if s <= cur_end + 1:
                    if e > cur_end:
                        cur_end = e
                else:
                    total += (cur_end - cur_start + 1)
                    cur_start, cur_end = s, e

            total += (cur_end - cur_start + 1)
            return int(total)

        # define col names of gtf file
        cols = ["seqname","source","feature","start","end","score","strand","frame","attribute"]
        gtf = pd.read_csv(
            gtf_path, sep="\t", comment="#", header=None, names=cols,
            dtype={"seqname":"category","source":"category","feature":"category",
                   "start":int,"end":int,"score":"string","strand":"category",
                   "frame":"string","attribute":"string"}
        )

        # keep only exons
        exons = gtf[gtf["feature"] == "exon"].copy()
        if exons.empty:
            return pd.DataFrame(columns=["gene_id","exonic_length"])

        # gene_id
        exons["gene_id"] = exons["attribute"].map(extract_gene_id)
        exons = exons.dropna(subset=["gene_id"])

        # for safety, ensure start <= end
        bad = (exons["end"] < exons["start"])
        if bad.any():
            exons.loc[bad, ["start","end"]] = exons.loc[bad, ["end","start"]].values

        # group by gene and merge intervals
        def _sum_for_gene(g):
            ivals = g[["start","end"]].to_numpy(dtype=np.int64)
            return merge_and_sum(ivals)

        lengths = (
            exons.groupby("gene_id", sort=False, observed=True)
                 .apply(_sum_for_gene)
                 .reset_index(name="exonic_length")
        )
        return lengths

def calculate_fpkm(expr_df, lengths_df, robust=True):
        """
        expr_df: DataFrame, rows=genes, cols=samples, values=raw counts
        lengths_df: DataFrame with columns ["gene_id", "gene_length"] (bp)

        Returns: FPKM DataFrame with same shape as expr_df
        if robus = True, returns size-factor normalized fpkms, same as DESEQ2
        """

        lengths = lengths_df.set_index("gene_id")["exonic_length"]
        expr_df = expr_df.copy()

        # make sure order matches
        expr_df = expr_df.loc[expr_df.index.intersection(lengths.index)]
        lengths = lengths.loc[expr_df.index]

        if robust is False:
            library_size = expr_df.sum(axis=0)
        else:
            library_size = self.size_factors.T * np.exp(np.mean(np.log(expr_df.sum(axis=0))))

        fpkm = expr_df.div(lengths, axis=0) * 1e9
        fpkm = fpkm.div(library_size, axis=1)

        return fpkm

In [4]:

sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")

or_res_path = "/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/py_outrider_runs/all_cohorts/oht_cov_diag_lr_0_0001_epoc200_gpu/or_variants.parquet"
needed_cols = ["sampleID", "zScore", "geneID",  "geneID_short", "PROTEIN_INT"]
# py_or_res_all = pd.read_parquet(or_res_path, columns=needed_cols)

py_or_res_all = pl.scan_parquet(or_res_path).select(needed_cols).collect().to_pandas()

py_or_res_all = py_or_res_all.merge(sa[["pid", "Diag", "Oncotree Code"]], left_on="sampleID", right_on="pid")

In [10]:
py_or_res_all.shape

(70972671, 8)

In [5]:
matrix = py_or_res_all.pivot(index='geneID', columns='sampleID', values='PROTEIN_INT').fillna(0)


In [67]:
gtf = "//omics/odcf/analysis/hipo/hipo_021/outlier_analysis/Data/gencode.v19.annotation_plain.gtf"
gene_lengths_df = gene_exonic_lengths_from_gtf(gtf)
fpkms = calculate_fpkm(matrix, gene_lengths_df, robust=False)


/tmp/ipykernel_477857/1631746104.py:72: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_sum_for_gene)


In [27]:
# example_counts = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/drop_runs/drop_master_202502_allGenes/processed_results/exported_counts/ACC--hg19--v19/geneCounts.tsv.gz", sep="\t")
# 2342 for ENSG00000000003.10 and sample STW65R, looks fine
                             

In [6]:
sf_df = pd.read_parquet(f"/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/py_outrider_runs/all_cohorts/oht_cov_diag_lr_0_0001_epoc200_gpu//size_factors.parquet")

In [7]:
# 1. Pivot the main data to a matrix (Genes x Samples)

# 2. Prepare the size factor Series 
# We need the sampleID to be the index to align with the matrix columns
sf_series = sf_df.set_index('sampleID')['size_factors']

# Ensure the size factors are in the same order as the matrix columns
sf_series = sf_series.reindex(matrix.columns)

normalized_counts = matrix.divide(sf_series, axis=1)

# 3. Log transformation and Z-score calculation
log2_matrix = np.log2(normalized_counts + 1)

# 4. Apply Size Factor Normalization
# matrix.divide(series, axis=1) aligns the SampleIDs automatically

zscores_sf_normalized = log2_matrix.apply(lambda x: (x - x.mean()) / x.std(), axis=1)

# 4. Apply Size Factor Normalization
# matrix.divide(series, axis=1) aligns the SampleIDs automatically
# zscores_sf_normalized = zscores.divide(sf_series, axis=1)

# 5. Convert back to Long Format
final_df = zscores_sf_normalized.reset_index().melt(
    id_vars='geneID', 
    var_name='sampleID', 
    value_name='sf_zScore'
)
mapping = py_or_res_all[['geneID', 'geneID_short']].drop_duplicates()
long_df = final_df.merge(mapping, on='geneID', how='left')

# 4. Calculate P-values
# We use the absolute Z-score to get a two-tailed p-value
long_df['pvalue'] = 2 * norm.cdf(-long_df['sf_zScore'].abs())

# 5. Calculate Adjusted P-values (Multiple Testing Correction)
# 'fdr_by' is the Benjamini-Yekutieli method from your R snippet
_, padj, _, _ = multipletests(long_df['pvalue'], method='fdr_by')
long_df['padjust'] = padj

# 6. Final Sort by Significance
long_df = long_df.sort_values('zScore')

In [8]:
long_df.shape

(70972671, 6)

In [9]:
long_df

,geneID,sampleID,sf_zScore,geneID_short,pvalue,padjust
20722559,ENSG00000100330.11,B171CQ9,-16.953350,ENSG00000100330,1.817923e-64,2.406924e-55
47214432,ENSG00000095261.9,NZJQV3,-16.713478,ENSG00000095261,1.045556e-62,6.921564e-54
40628418,ENSG00000198954.4,LBPCK5,-15.070091,ENSG00000198954,2.547625e-51,1.124349e-42
40044741,ENSG00000149179.9,L4C4GQC,-14.970852,ENSG00000149179,1.138534e-50,3.768537e-42
20726559,ENSG00000132952.7,B171CQ9,-14.662232,ENSG00000132952,1.125135e-48,2.979349e-40
...,...,...,...,...,...,...
60415757,ENSG00000125812.11,V7PGJH,11.426366,ENSG00000125812,3.087630e-30,5.241041e-23
23494497,ENSG00000111605.12,CB5S4F,11.729306,ENSG00000111605,9.019511e-32,2.211445e-24
10838775,ENSG00000139620.8,6ERY1Z,12.319336,ENSG00000139620,7.127910e-35,2.859797e-27
992122,ENSG00000167699.9,1HCWLP,12.845256,ENSG00000167699,9.144899e-38,6.726564e-30


In [11]:
long_df.to_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/py_outrider_runs/pyoutrider_zscores/aggregated_sf_zScores_all.parquet", index=None)